# Kentucky Weather → LSTM Evaluation
**Goal**: Engineer the exact 45 features the pre-trained Indiana LSTM model expects, using the Kentucky weather dataset, then run inference.

### ⚠️ Two known limitations
| Issue | Impact |
|---|---|
| Kentucky CSV columns are **mislabeled** (shuffled) — corrected in Section 3 | Without fixing this, every derived feature would be wrong |
| **25 of 45 features** will be zero-filled (13 energy lags + 12 indoor/UV sensor columns not in dataset) | Predictions will be degraded but the model will still run |

### The 45 model features — what we can/can't supply
| Category | Features | Source |
|---|---|---|
| Weather (direct) | `out_temp`, `pt_dew`, `hum_out`, `bar`, `speed_wind` | Kentucky CSV (after relabeling) |
| Derived from datetime/weather | `hour`, `hour_sin/cos`, `day_of_week`, `day_of_week_sin`, `month`, `month_sin/cos`, `day_of_year`, `is_weekend`, `is_raining`, `temp_x_hum`, `heat_index_approx`, `wind_chill_approx`, `density_in_air` | Computed |
| Energy lags/rolling/EWM | `lag_1..48`, `rolling_mean/std_6/12/24`, `ewm_alpha_02/05` | ❌ Zero-filled |
| Indoor/UV sensors | `temp_in`, `hum_in`, `dew_in`, `emc_in`, `temp_hi`, `temp_low`, `speed_hi`, `run_wind`, `index_thsw`, `index_uv`, `uv_load`, `dose_uv` | ❌ Zero-filled |


## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")


Libraries loaded.


## 2. Configuration

In [3]:
# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
KENTUCKY_WEATHER_PATH = '../data/processed/kentucky_weather_2005_2018.csv'
FEATURE_SCALER_PATH   = '../../01_campus_substation/models/feature_scaler.pkl'
TARGET_SCALER_PATH    = '../../01_campus_substation/models/target_scaler.pkl'
MODEL_PATH            = '../../01_campus_substation/models/best_lstm_model.keras'          # your saved .keras / .h5 model

OUTPUT_CSV            = '../data/processed/kentucky_features_engineered.csv'
# ────────────────────────────────────────────────────────────────────────────

SEQ_LEN = 168   # 1-week look-back (must match training)

# Exact feature order the RobustScaler was fitted on (45 features)
FEATURE_COLS = [
    'ewm_alpha_05', 'lag_1', 'ewm_alpha_02', 'lag_2', 'rolling_mean_6',
    'lag_3', 'rolling_std_12', 'rolling_mean_24', 'out_temp', 'index_thsw',
    'rolling_std_24', 'rolling_std_6', 'rolling_mean_12', 'lag_24', 'pt_dew',
    'hour_cos', 'density_in_air', 'lag_48', 'hum_out', 'index_uv',
    'temp_in', 'uv_load', 'month_sin', 'temp_x_hum', 'month_cos',
    'speed_wind', 'speed_hi', 'hour_sin', 'bar', 'temp_low',
    'heat_index_approx', 'dose_uv', 'hour', 'is_weekend', 'temp_hi',
    'day_of_year', 'day_of_week', 'month', 'hum_in', 'emc_in',
    'wind_chill_approx', 'day_of_week_sin', 'is_raining', 'run_wind', 'dew_in'
]

assert len(FEATURE_COLS) == 45
print(f"Feature count: {len(FEATURE_COLS)} ✅")


Feature count: 45 ✅


## 3. Load & Fix Mislabeled Columns

The Kentucky CSV headers are **shifted/scrambled** relative to the actual data.
The table below shows the correction applied:

| CSV label (wrong) | Actual variable | Evidence |
|---|---|---|
| `Temp_C` | UV Index | Only integers 0–10 (11 unique values) |
| `Dew_Point_C` | Temperature °C | Range −29 to 37 °C — matches Kentucky climate |
| `RH_percent` | Dew Point °C | Range −33 to 27 °C — valid dew points |
| `Pressure_kPa` | Relative Humidity % | Range 8–100 % |
| `Wind_Dir_deg` | Pressure kPa | Range 94–102 kPa — sea-level pressure ✅ |
| `Wind_Speed_mps` | Wind Direction ° | Range 0–360° |
| `Visibility_km` | Wind Speed m/s | Range 0–35.7 m/s (0–128 km/h) |
| `Precip_cm` | Precipitation cm | Unchanged — 0–36 cm, mostly zero ✅ |


In [4]:
raw = pd.read_csv(KENTUCKY_WEATHER_PATH, parse_dates=['datetime'])
raw['datetime'] = raw['datetime'].dt.round('h')
raw = raw.sort_values('datetime').reset_index(drop=True)

print("Raw shape:", raw.shape)
print("Raw columns:", raw.columns.tolist())
print()
print(raw.head(3).to_string())


Raw shape: (166560, 9)
Raw columns: ['datetime', 'Temp_C', 'Dew_Point_C', 'RH_percent', 'Pressure_kPa', 'Wind_Dir_deg', 'Wind_Speed_mps', 'Visibility_km', 'Precip_cm']

             datetime  Temp_C  Dew_Point_C  RH_percent  Pressure_kPa  Wind_Dir_deg  Wind_Speed_mps  Visibility_km  Precip_cm
0 2005-01-01 00:00:00    10.0          0.0         0.0         100.0          98.6             0.0            4.2        0.0
1 2005-01-01 01:00:00    10.0          0.0         0.0         100.0          98.6            20.0            5.2        0.0
2 2005-01-01 02:00:00    10.0          0.0         0.0         100.0          98.6             0.0            4.7        0.0


In [5]:
# Apply the correct column mapping
df = pd.DataFrame()
df['datetime']    = raw['datetime']
df['out_temp']    = raw['Dew_Point_C']    # actual temperature
df['pt_dew']      = raw['RH_percent']     # actual dew point
df['hum_out']     = raw['Pressure_kPa']   # actual RH %
df['bar']         = raw['Wind_Dir_deg']   # actual pressure kPa
df['speed_wind']  = raw['Visibility_km']  # actual wind speed m/s
df['wind_dir']    = raw['Wind_Speed_mps'] # actual wind direction (not a model feature)
df['index_uv_raw']= raw['Temp_C']         # UV index integers 0-10
df['precip_cm']   = raw['Precip_cm']      # precipitation

print("Corrected column stats:")
for col in ['out_temp','pt_dew','hum_out','bar','speed_wind']:
    s = df[col]
    print(f"  {col:15s}  min={s.min():7.2f}  max={s.max():7.2f}  mean={s.mean():7.2f}")


Corrected column stats:
  out_temp         min= -29.40  max=  37.20  mean=  12.47
  pt_dew           min= -32.80  max=  27.20  mean=   6.59
  hum_out          min=   8.00  max= 100.00  mean=  69.90
  bar              min=  94.30  max= 101.60  mean=  98.25
  speed_wind       min=   0.00  max=  35.70  mean=   4.16


## 4. Clean Missing Values

In [6]:
df = df.interpolate(method='linear', limit=3).ffill().bfill()
assert df.isnull().sum().sum() == 0, "NaNs remain!"
print(f"Shape: {df.shape}  |  Missing values: 0 ✅")
print(f"Date range: {df['datetime'].min()}  →  {df['datetime'].max()}")


Shape: (166560, 9)  |  Missing values: 0 ✅
Date range: 2005-01-01 00:00:00  →  2023-12-31 23:00:00


## 5. Feature Engineering
### 5a. Datetime Features

In [7]:
dt = df['datetime']

df['hour']           = dt.dt.hour
df['hour_sin']       = np.sin(2 * np.pi * dt.dt.hour       / 24)
df['hour_cos']       = np.cos(2 * np.pi * dt.dt.hour       / 24)
df['day_of_week']    = dt.dt.dayofweek
df['day_of_week_sin']= np.sin(2 * np.pi * dt.dt.dayofweek  / 7)
df['month']          = dt.dt.month
df['month_sin']      = np.sin(2 * np.pi * dt.dt.month      / 12)
df['month_cos']      = np.cos(2 * np.pi * dt.dt.month      / 12)
df['day_of_year']    = dt.dt.dayofyear
df['is_weekend']     = (dt.dt.dayofweek >= 5).astype(int)

print("Datetime features created ✅")
print(df[['hour','hour_sin','hour_cos','day_of_week','month','is_weekend']].head(3))


Datetime features created ✅
   hour  hour_sin  hour_cos  day_of_week  month  is_weekend
0     0  0.000000  1.000000            5      1           1
1     1  0.258819  0.965926            5      1           1
2     2  0.500000  0.866025            5      1           1


### 5b. Weather-Derived Features

In [8]:
# Precipitation flag
df['is_raining'] = (df['precip_cm'] > 0).astype(int)

# Temperature × Humidity interaction
df['temp_x_hum'] = df['out_temp'] * df['hum_out']

# Air density  ρ = P / (R_specific × T_kelvin)  [kg/m³]
R_specific = 287.05  # J/(kg·K) for dry air
df['density_in_air'] = (df['bar'] * 1000) / (R_specific * (df['out_temp'] + 273.15))

# Heat Index (Rothfusz regression, °C)
# Applied unconditionally — model was trained this way regardless of temp/RH threshold
T  = df['out_temp']
RH = df['hum_out']
Tf = T * 9/5 + 32   # convert to °F for formula
HI_f = (-42.379
        + 2.04901523  * Tf
        + 10.14333127 * RH
        - 0.22475541  * Tf * RH
        - 0.00683783  * Tf**2
        - 0.05481717  * RH**2
        + 0.00122874  * Tf**2 * RH
        + 0.00085282  * Tf   * RH**2
        - 0.00000199  * Tf**2 * RH**2)
df['heat_index_approx'] = (HI_f - 32) * 5/9   # back to °C

# Wind Chill (Environment Canada formula, °C)
# Applied unconditionally for same reason
V_kmh = df['speed_wind'] * 3.6
# Avoid log/power issues when wind is 0
V_safe = V_kmh.clip(lower=0.01)
df['wind_chill_approx'] = (13.12
                           + 0.6215  * T
                           - 11.37   * V_safe**0.16
                           + 0.3965  * T * V_safe**0.16)

print("Weather-derived features created ✅")
cols_check = ['is_raining','temp_x_hum','density_in_air','heat_index_approx','wind_chill_approx']
print(df[cols_check].describe().round(3))


Weather-derived features created ✅
       is_raining  temp_x_hum  density_in_air  heat_index_approx  \
count  166560.000  166560.000      166560.000         166560.000   
mean        0.056     836.522           1.200             42.436   
std         0.231     738.114           0.048             30.339   
min         0.000   -2536.500           1.096             13.131   
25%         0.000     280.800           1.162             24.257   
50%         0.000     890.000           1.191             28.985   
75%         0.000    1450.800           1.234             51.554   
max         1.000    2696.600           1.423            340.169   

       wind_chill_approx  
count         166560.000  
mean              11.195  
std               13.098  
min              -40.359  
25%                1.005  
50%               12.943  
75%               21.877  
max               41.689  


### 5c. Zero-Filled Features
These 25 features are unavailable and will be set to **0.0**.

**Energy features (13)** — `lag_1`, `lag_2`, `lag_3`, `lag_24`, `lag_48`,
`rolling_mean_6`, `rolling_std_6`, `rolling_mean_12`, `rolling_std_12`,
`rolling_mean_24`, `rolling_std_24`, `ewm_alpha_02`, `ewm_alpha_05`

**Indoor / UV sensor features (12)** — `temp_in`, `hum_in`, `dew_in`, `emc_in`,
`temp_hi`, `temp_low`, `speed_hi`, `run_wind`, `index_thsw`, `index_uv`,
`uv_load`, `dose_uv`


In [9]:
ZERO_FILL_COLS = [
    # Energy lags
    'lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_48',
    # Rolling stats on energy
    'rolling_mean_6', 'rolling_std_6',
    'rolling_mean_12', 'rolling_std_12',
    'rolling_mean_24', 'rolling_std_24',
    # EWM on energy
    'ewm_alpha_02', 'ewm_alpha_05',
    # Indoor sensors
    'temp_in', 'hum_in', 'dew_in', 'emc_in',
    # Station-specific
    'temp_hi', 'temp_low', 'speed_hi', 'run_wind',
    'index_thsw', 'index_uv', 'uv_load', 'dose_uv',
]

for col in ZERO_FILL_COLS:
    df[col] = 0.0

print(f"{len(ZERO_FILL_COLS)} columns zero-filled ✅")


25 columns zero-filled ✅


## 6. Validate — All 45 Features Present & Ordered

In [10]:
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("✅ All 45 feature columns present")
print()
print(f"{'#':>3}  {'Feature':<22}  {'Source'}")
print("-" * 55)
available = set(df.columns)
zero_set  = set(ZERO_FILL_COLS)
for i, col in enumerate(FEATURE_COLS, 1):
    if col in zero_set:
        src = "⚠️  zero-filled"
    else:
        src = "✅ computed"
    print(f"{i:>3}. {col:<22}  {src}")


✅ All 45 feature columns present

  #  Feature                 Source
-------------------------------------------------------
  1. ewm_alpha_05            ⚠️  zero-filled
  2. lag_1                   ⚠️  zero-filled
  3. ewm_alpha_02            ⚠️  zero-filled
  4. lag_2                   ⚠️  zero-filled
  5. rolling_mean_6          ⚠️  zero-filled
  6. lag_3                   ⚠️  zero-filled
  7. rolling_std_12          ⚠️  zero-filled
  8. rolling_mean_24         ⚠️  zero-filled
  9. out_temp                ✅ computed
 10. index_thsw              ⚠️  zero-filled
 11. rolling_std_24          ⚠️  zero-filled
 12. rolling_std_6           ⚠️  zero-filled
 13. rolling_mean_12         ⚠️  zero-filled
 14. lag_24                  ⚠️  zero-filled
 15. pt_dew                  ✅ computed
 16. hour_cos                ✅ computed
 17. density_in_air          ✅ computed
 18. lag_48                  ⚠️  zero-filled
 19. hum_out                 ✅ computed
 20. index_uv                ⚠️  zero-filled

## 7. Apply the Saved RobustScaler

In [11]:
feature_scaler = joblib.load(FEATURE_SCALER_PATH)
target_scaler  = joblib.load(TARGET_SCALER_PATH)

# Verify the scaler expects exactly our 45-column order
assert list(feature_scaler.feature_names_in_) == FEATURE_COLS, \
    "Column order mismatch with scaler! Check FEATURE_COLS."

X_raw    = df[FEATURE_COLS].values
X_scaled = feature_scaler.transform(X_raw)

print(f"X_scaled shape : {X_scaled.shape}")
print(f"X_scaled mean  : {X_scaled.mean():.4f}")
print(f"X_scaled std   : {X_scaled.std():.4f}")
print("Scaler applied ✅")


X_scaled shape : (166560, 45)
X_scaled mean  : 48.3070
X_scaled std   : 275.8564
Scaler applied ✅


## 8. Create LSTM Sequences (look-back = 168 h)

In [12]:
def create_sequences(X, seq_len):
    xs = []
    for i in range(seq_len, len(X)):
        xs.append(X[i - seq_len : i])
    return np.array(xs)

X_seq     = create_sequences(X_scaled, SEQ_LEN)
seq_dates = df['datetime'].values[SEQ_LEN:]

print(f"Sequence array : {X_seq.shape}  →  (samples, timesteps, features)")
print(f"Date range     : {pd.Timestamp(seq_dates[0])}  →  {pd.Timestamp(seq_dates[-1])}")


MemoryError: Unable to allocate 9.37 GiB for an array with shape (166392, 168, 45) and data type float64

## 9. Run Inference

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

y_pred_scaled = model.predict(X_seq, batch_size=512, verbose=1)
y_pred        = target_scaler.inverse_transform(y_pred_scaled).flatten()

print(f"\nPredictions shape : {y_pred.shape}")
print(f"Predicted range   : {y_pred.min():.1f}  →  {y_pred.max():.1f} MW")


## 10. Evaluate (if you have Kentucky energy data)

In [ ]:
# ── Un-comment if you have Kentucky energy consumption ───────────────────────
# KENTUCKY_ENERGY_PATH = 'kentucky_energy.csv'   # needs 'datetime' + energy MW column
#
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
#
# energy_ky = pd.read_csv(KENTUCKY_ENERGY_PATH, parse_dates=['datetime'])
# energy_ky['datetime'] = energy_ky['datetime'].dt.round('h')
# energy_ky = energy_ky.set_index('datetime')
#
# TARGET_COL = 'aep_mw'   # ← change to your column name
# y_true = energy_ky.loc[pd.DatetimeIndex(seq_dates), TARGET_COL].values
#
# mae  = mean_absolute_error(y_true, y_pred)
# rmse = np.sqrt(mean_squared_error(y_true, y_pred))
# mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
# r2   = r2_score(y_true, y_pred)
#
# print('=' * 45)
# print('  KENTUCKY EVALUATION METRICS')
# print('=' * 45)
# print(f'  MAE  : {mae:>10,.2f} MW')
# print(f'  RMSE : {rmse:>10,.2f} MW')
# print(f'  MAPE : {mape:>10.2f} %')
# print(f'  R2   : {r2:>10.4f}')
# print('=' * 45)

print("Un-comment the block above once you have Kentucky energy data.")


## 11. Save Outputs

In [ ]:
# Save engineered feature matrix
df[['datetime'] + FEATURE_COLS].to_csv(OUTPUT_CSV, index=False)
print(f"Feature matrix saved → {OUTPUT_CSV}")

# Save predictions
results = pd.DataFrame({'datetime': seq_dates, 'pred_mw': y_pred})
results.to_csv('../outputs/kentucky_predictions.csv', index=False)
print(f"Predictions saved   → kentucky_predictions.csv")

# Save sequences for re-use
np.save('../data/processed/X_kentucky_seq.npy', X_seq)
print(f"Sequences saved     → X_kentucky_seq.npy  {X_seq.shape}")
